In [1]:
import pandas as pd

orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")
order_products = pd.read_csv("order_products__prior.csv", nrows=500)

print(order_products.head())

   order_id  product_id  add_to_cart_order  reordered
0         2       33120                  1          1
1         2       28985                  2          1
2         2        9327                  3          0
3         2       45918                  4          1
4         2       30035                  5          0


In [2]:
data = pd.merge(order_products, products, on="product_id")

print(data[['order_id','product_name']].head())

   order_id           product_name
0         2     Organic Egg Whites
1         2  Michigan Organic Kale
2         2          Garlic Powder
3         2         Coconut Butter
4         2      Natural Sweetener


In [3]:
transactions = data.groupby('order_id')['product_name'].apply(list)

print(transactions.head())

order_id
2    [Organic Egg Whites, Michigan Organic Kale, Ga...
3    [Total 2% with Strawberry Lowfat Greek Straine...
4    [Plain Pre-Sliced Bagels, Honey/Lemon Cough Dr...
5    [Bag of Organic Bananas, Just Crisp, Parmesan,...
6    [Cleanse, Dryer Sheets Geranium Scent, Clean D...
Name: product_name, dtype: object


In [4]:
from itertools import combinations
from collections import Counter

pair_counts = Counter()

for items in transactions:
    pairs = combinations(items, 2)
    pair_counts.update(pairs)

print(pair_counts.most_common(10))

[(('Bag of Organic Bananas', 'Organic Raspberries'), 2), (('Banana', 'Organic Avocado'), 2), (('Bag of Organic Bananas', 'Organic Tomato Cluster'), 2), (('Organic Egg Whites', 'Michigan Organic Kale'), 1), (('Organic Egg Whites', 'Garlic Powder'), 1), (('Organic Egg Whites', 'Coconut Butter'), 1), (('Organic Egg Whites', 'Natural Sweetener'), 1), (('Organic Egg Whites', 'Carrots'), 1), (('Organic Egg Whites', 'Original Unflavored Gelatine Mix'), 1), (('Organic Egg Whites', 'All Natural No Stir Creamy Almond Butter'), 1)]


In [5]:
def recommend(product, pair_counts, top_n=5):
    recommendations = []
    
    for (p1, p2), count in pair_counts.items():
        if p1 == product:
            recommendations.append((p2, count))
        elif p2 == product:
            recommendations.append((p1, count))
            
    recommendations = sorted(recommendations, key=lambda x: x[1], reverse=True)
    
    return recommendations[:top_n]

In [6]:
recommend("Bag of Organic Bananas", pair_counts)

[('Organic Raspberries', 2),
 ('Organic Tomato Cluster', 2),
 ('Just Crisp, Parmesan', 1),
 ('Fresh Fruit Salad', 1),
 ('2% Reduced Fat Milk', 1)]